# 19. 공식 라벨만으로 만든 급등 국면 검증 세트
- 정답: **공식 08-25 라벨** (666채널, 정답 구간 08-25~09-12 = 8월 말 급등 + 9월 초 → 본선과 같은 국면)
- 전주 라벨 역할: **공식 08-11 라벨** (본선의 08-25 라벨 역할)
- 차이: 간격 14일(본선 7일), 정답 구간 18일(본선 26일) → 전주 라벨의 힘은 본선보다 약할 것. 구조 비교용.
- 모델 학습: 자체 FUT18 라벨 07-26~08-07 (정답 구간이 08-25 이전에 끝남 → 누수 없음)

In [1]:
# ===== 라이브러리 =====
import os, glob, platform, warnings
import duckdb                                   # 큰 CSV를 메모리에 다 올리지 않고 SQL로 집계
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import rankdata                # 값 → 순위 변환 (블렌딩용)
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier   # 비교용 트리 모델
from sklearn.metrics import average_precision_score, roc_auc_score   # PR-AUC(주 지표), ROC-AUC(보조)

warnings.filterwarnings('ignore')
# 한글 폰트: OS별로 자동 선택 (윈도우 / 맥 / 그 외)
plt.rcParams['font.family'] = {'Windows': 'Malgun Gothic', 'Darwin': 'AppleGothic'}.get(platform.system(), 'NanumGothic')
plt.rcParams['axes.unicode_minus'] = False      # 음수 부호 깨짐 방지
pd.set_option('display.max_columns', 50)

# ===== 경로 · 상수 =====
RAW = '../data/raw/epoch_data'   # 배포 데이터 폴더
D = '2026-09-01'                 # 본선 기준일
FUT = 26                         # 본선 정답 구간 길이 (일)
SEED = 42

# ===== CSV 전부를 파일 이름 그대로 DuckDB 뷰로 등록 (예: videos.csv → videos) =====
con = duckdb.connect()
for p in sorted(glob.glob(f'{RAW}/*.csv')):
    name = os.path.splitext(os.path.basename(p))[0]
    con.sql(f"CREATE OR REPLACE VIEW {name} AS SELECT * FROM read_csv('{p}', header=true, auto_detect=true)")

# 데이터 범위 확인: 업로드 시각이 기준일(09-01) 전에서 끝나는지
# con.sql("SELECT MIN(published_at), MAX(published_at), COUNT(*) FROM video_5d_views").df()

In [2]:
def make_features(Dref):
    """기준일 Dref 이전(published_at < Dref) 영상만으로 채널별 피처를 만든다. 학습·예측 모두 이 함수 하나로."""
    f = con.sql(f"""
    -- lv: 기준일 이전 롱폼 영상 중 view_5d가 있는 것 + 기준일로부터 며칠 전 영상인지
    WITH lv AS (
        SELECT channel_id, view_5d, like_5d, comment_5d,
               DATE_DIFF('hour', published_at, TIMESTAMP '{Dref}') / 24.0 AS days_ago
        FROM video_5d_views
        WHERE is_shorts = 0 AND view_5d IS NOT NULL AND published_at < TIMESTAMP '{Dref}'
    ),
    -- v5: 롱폼 성과 요약 (수준 · 최근 구간 · 반응 · 변동성 · 추세)
    v5 AS (
        SELECT channel_id,
               COUNT(*)                                        AS n_long_5d,        -- 측정된 롱폼 수
               MEDIAN(view_5d)                                 AS past_med,         -- = 정답 계산식의 past_score
               MEDIAN(view_5d) FILTER (WHERE days_ago < 7)     AS med_7,            -- 최근 7일 중앙값
               MEDIAN(view_5d) FILTER (WHERE days_ago < 14)    AS med_14,           -- 최근 14일 중앙값
               MEDIAN(view_5d) FILTER (WHERE days_ago >= 14)   AS med_old,          -- 14일 이전 중앙값
               MEDIAN(like_5d / NULLIF(view_5d, 0))            AS like_rate,        -- 조회수 대비 좋아요
               MEDIAN(comment_5d / NULLIF(view_5d, 0))         AS comment_rate,     -- 조회수 대비 댓글
               STDDEV(LN(1 + view_5d))                         AS log_view_std,     -- 영상 간 편차 (터지는 채널인지)
               QUANTILE_CONT(LN(1 + view_5d), 0.1)             AS log_q10,          -- 하위 10% 영상 수준
               QUANTILE_CONT(LN(1 + view_5d), 0.9)             AS log_q90,          -- 상위 10% 영상 수준
               REGR_SLOPE(LN(1 + view_5d), -days_ago)          AS trend_slope,      -- 시간에 따른 조회수 기울기
               MIN(days_ago)                                   AS days_since_meas   -- 마지막 측정 영상이 며칠 전인지
        FROM lv GROUP BY 1
    ),
    -- sv: 쇼츠 성과 (롱폼 대비 비교용)
    sv AS (
        SELECT channel_id, MEDIAN(view_5d) AS short_med
        FROM video_5d_views
        WHERE is_shorts = 1 AND view_5d IS NOT NULL AND published_at < TIMESTAMP '{Dref}'
        GROUP BY 1
    ),
    -- up: 업로드 활동 (view_5d가 없어도 업로드 시각은 알 수 있으므로 videos 테이블 사용)
    up AS (
        SELECT channel_id,
               COUNT(*) FILTER (WHERE is_shorts = 0 AND published_at >= TIMESTAMP '{Dref}' - INTERVAL 7 DAY)  AS n_long_7,
               COUNT(*) FILTER (WHERE is_shorts = 0 AND published_at >= TIMESTAMP '{Dref}' - INTERVAL 14 DAY) AS n_long_14,
               COUNT(*) FILTER (WHERE is_shorts = 0 AND published_at >= TIMESTAMP '{Dref}' - INTERVAL 28 DAY) AS n_long_28,
               COUNT(*) FILTER (WHERE is_shorts = 1 AND published_at >= TIMESTAMP '{Dref}' - INTERVAL 14 DAY) AS n_short_14,
               AVG(duration_sec) FILTER (WHERE is_shorts = 0)                                                  AS avg_dur,          -- 롱폼 평균 길이(초)
               DATE_DIFF('hour', MAX(published_at) FILTER (WHERE is_shorts = 0), TIMESTAMP '{Dref}') / 24.0  AS days_since_long   -- 마지막 롱폼 후 경과일
        FROM videos WHERE published_at < TIMESTAMP '{Dref}' GROUP BY 1
    ),
    -- games: 영상에 매핑된 Steam 게임 수 (매핑 없으면 0)
    games AS (
        SELECT v.channel_id, COUNT(DISTINCT vg.app_id) AS n_games
        FROM videos v LEFT JOIN video_games vg USING (video_id)
        WHERE v.published_at < TIMESTAMP '{Dref}' GROUP BY 1
    )
    SELECT * FROM v5 LEFT JOIN sv USING (channel_id) LEFT JOIN up USING (channel_id)
                     LEFT JOIN games USING (channel_id)
    """).df()

    # ===== 파생 피처 (라벨이 log 차이이므로 log 스케일로 맞춤) =====
    lp = np.log1p(f.past_med)
    f['log_past_med']      = lp                                            # 과거 수준 (낮을수록 Rising 쉬움)
    f['mom_7']             = np.log1p(f.med_7) - lp                        # 최근 7일 vs 전체
    f['mom_14']            = np.log1p(f.med_14) - lp                       # 최근 14일 vs 전체
    f['mom_old']           = np.log1p(f.med_14) - np.log1p(f.med_old)     # 최근 14일 vs 그 이전
    f['short_ratio']       = f.n_short_14 / (f.n_short_14 + f.n_long_14).replace(0, np.nan)   # 최근 업로드 중 쇼츠 비중
    f['log_short_med_rel'] = np.log1p(f.short_med) - lp                    # 쇼츠 성과 vs 롱폼 성과
    f['has_game']          = (f.n_games > 0).astype(int)                   # 게임 채널 여부
    f['log_n_long_5d']     = np.log1p(f.n_long_5d)
    return f

# 모델에 넣는 피처 23개 (구독자 피처는 분포 차이로 제외 — 위 설명 참고)
FEATURES = ['log_past_med', 'n_long_5d', 'like_rate', 'n_long_14', 'n_short_14', 'short_ratio', 'days_since_long',
            'mom_14', 'log_view_std', 'n_games', 'has_game', 'mom_7', 'mom_old', 'comment_rate',
            'log_q10', 'log_q90', 'trend_slope', 'days_since_meas', 'n_long_7', 'n_long_28', 'avg_dur',
            'log_short_med_rel', 'log_n_long_5d']
print(len(FEATURES), 'features')
make_features(D)[['channel_id'] + FEATURES].head()   # 09-01 기준으로 잘 만들어지는지 확인

23 features


,channel_id,log_past_med,n_long_5d,like_rate,n_long_14,n_short_14,short_ratio,days_since_long,mom_14,log_view_std,n_games,has_game,mom_7,mom_old,comment_rate,log_q10,log_q90,trend_slope,days_since_meas,n_long_7,n_long_28,avg_dur,log_short_med_rel,log_n_long_5d
0,UCuWDza8A_ntbjwBZgW0ZSNw,9.461838,22,0.005676,5,19,0.791667,2.875000,0.013939,0.759894,0,0,0.405608,0.028074,0.001430,8.860442,10.902154,0.004554,2.875000,2,10,491.576923,1.776098,3.135494
1,UCzps05UZ7XpX3SFoFyYIDow,10.101026,55,0.006191,19,1,0.050000,0.833333,-0.092211,0.884408,1,1,0.488584,-0.092211,0.002694,9.026393,11.315231,0.004092,0.833333,12,37,1525.346667,1.415752,4.025352
2,UCTNyl3P7q6PNhPddh6d7WJQ,3.135494,8,0.066316,2,4,0.666667,1.083333,7.214280,3.136847,1,1,7.214280,7.577186,0.000000,2.458803,8.580298,0.111421,1.083333,2,4,2631.250000,4.596217,2.197225
3,UCcJliZnsRbNw4I4uWSGq_wA,8.737453,25,0.001551,17,0,0.000000,0.208333,0.904086,1.808256,0,0,1.071010,3.037594,0.000000,6.158616,10.410077,0.141866,0.208333,15,23,9943.040000,NaN,3.258097
4,UCQzgMwT5Q6x-Mtq65HRS_0Q,8.667164,23,0.007524,8,2,0.200000,0.541667,0.525929,2.811501,0,0,0.525929,0.905312,0.000924,5.610890,9.943376,0.046821,0.541667,4,16,1330.259259,0.341978,3.178054


In [3]:
def make_labels(Dref, fut=FUT):
    """공식 라벨 규칙 재현: 기준일 Dref 이후 fut일 동안의 성장이 상위 20%면 1."""
    g = con.sql(f"""
        -- agg: 채널별 past_score(기준일 전) · future_score(기준일 ~ +fut일) — 롱폼만
        WITH agg AS (
          SELECT channel_id,
                 MEDIAN(view_5d) FILTER (WHERE published_at < TIMESTAMP '{Dref}') AS past_score,
                 COUNT(view_5d)  FILTER (WHERE published_at < TIMESTAMP '{Dref}') AS n_past,
                 COALESCE(MEDIAN(view_5d) FILTER (
                     WHERE published_at >= TIMESTAMP '{Dref}'
                       AND published_at <  TIMESTAMP '{Dref}' + INTERVAL {fut} DAY), 0) AS future_score   -- 미래 영상 없으면 0
          FROM video_5d_views
          WHERE is_shorts = 0 AND view_5d IS NOT NULL
          GROUP BY 1
        ),
        -- scored: 표본 조건(past 3편 이상, past_score ≥ 100) 통과 채널의 성장 점수
        scored AS (
          SELECT channel_id, LN(1 + future_score) - LN(1 + past_score) AS growth
          FROM agg WHERE n_past >= 3 AND past_score >= 100
        )
        -- 같은 기준일 안에서 상위 20% → 1
        SELECT channel_id,
               CASE WHEN growth >= QUANTILE_CONT(growth, 0.8) OVER () THEN 1 ELSE 0 END AS target
        FROM scored
    """).df()
    g['ref_date'] = Dref
    return g


def attach_features(labels):
    """라벨 표의 각 기준일마다, 그 기준일 시점 피처를 붙인다 (기준일별로 make_features 호출)."""
    return pd.concat([labels[labels.ref_date == d].merge(make_features(d), on='channel_id', how='left')
                      for d in sorted(labels.ref_date.unique())], ignore_index=True)




In [4]:
def logreg(C=0.1):
    """결측 → 중앙값 대체, 표준화, 로지스틱 회귀 (양성 20% 불균형은 class_weight로 보정)"""
    return Pipeline([('imp', SimpleImputer(strategy='median')),
                     ('sc', StandardScaler()),
                     ('m', LogisticRegression(max_iter=2000, C=C, class_weight='balanced'))])

def rank01(x):
    """값을 0~1 순위로 변환 (결측은 중앙값으로 채움)"""
    x = pd.Series(np.asarray(x, dtype=float))
    return rankdata(x.fillna(x.median())) / len(x)

def blend(proba, df):
    """모델 확률 순위 + 과거 중앙값 역순위의 평균 (PR-AUC는 순위만 봄)"""
    return (rank01(proba) + rank01(-df.log_past_med)) / 2

# 시간 분리: 라벨 창이 08-25 전에 끝나는 기준일로만 학습, 공식 08-25 라벨로 평가


In [5]:
from scipy.stats import rankdata, spearmanr
DROP = ['comment_rate', 'avg_dur', 'log_q90', 'like_rate', 'log_q10', 'trend_slope', 'has_game']
PRUNED = [f for f in FEATURES if f not in DROP]
TR_DATES = pd.date_range('2026-07-26', '2026-08-07', freq='D').strftime('%Y-%m-%d').tolist()
tr = attach_features(pd.concat([make_labels(d, fut=18) for d in TR_DATES], ignore_index=True))
tl = pd.read_csv(f'{RAW}/train_labels.csv')
tl['channel_id'] = tl.row_id.str.rsplit('_', n=1).str[0]; tl['ref_date'] = tl.row_id.str.rsplit('_', n=1).str[1]
va = tl[tl.ref_date == '2026-08-25'][['channel_id', 'target']].merge(make_features('2026-08-25'), on='channel_id', how='left')
va = va.merge(tl[tl.ref_date == '2026-08-11'][['channel_id', 'target']].rename(columns={'target': 'prev'}), on='channel_id', how='left')
y = va.target.values; N = len(va)
r = lambda x: rankdata(pd.Series(np.asarray(x, float)).fillna(np.nanmedian(x))) / N
p_hist = logreg(0.01).fit(tr[PRUNED], tr.target).predict_proba(va[PRUNED])[:, 1]
prev_f = va.prev.fillna(0.2).values
print('검증', N, '행 | 전주(08-11) 라벨 1/0/없음:', int((va.prev == 1).sum()), int((va.prev == 0).sum()), int(va.prev.isna().sum()))
print('전주 라벨별 Rising 비율:', va.groupby(va.prev.fillna(-1)).target.mean().round(3).to_dict())
ap = lambda s: round(average_precision_score(y, s), 4)
rule = r(prev_f + 1e-3 * r(-va.log_past_med))
res = {'모델 단독 (채널 이력 LR)': ap(p_hist), '규칙 단독 (전주라벨 + 저과거)': ap(rule), '전주 라벨만': ap(prev_f)}
for w in [0.3, 0.5, 0.7, 0.85]:
    res[f'모델 {1-w:.2f} + 규칙 {w}'] = ap((1 - w) * r(p_hist) + w * rule)
res['그룹 우선 + 그룹 안 모델 순서'] = ap(prev_f + 0.5 * r(p_hist))
res['그룹 우선 + 그룹 안 저과거 순서'] = ap(prev_f + 0.5 * r(-va.log_past_med))
res['그룹 우선 + 그룹 안 0.5모델+0.5저과거'] = ap(prev_f + 0.5 * (0.5 * r(p_hist) + 0.5 * r(-va.log_past_med)))
pd.Series(res).to_frame('PR-AUC (공식 08-25)')

검증 666 행 | 전주(08-11) 라벨 1/0/없음: 121 479 66
전주 라벨별 Rising 비율: {-1.0: 0.273, 0.0: 0.159, 1.0: 0.331}


,PR-AUC (공식 08-25)
모델 단독 (채널 이력 LR),0.4005
규칙 단독 (전주라벨 + 저과거),0.3891
전주 라벨만,0.2545
모델 0.70 + 규칙 0.3,0.4173
모델 0.50 + 규칙 0.5,0.4172
모델 0.30 + 규칙 0.7,0.4020
모델 0.15 + 규칙 0.85,0.3928
그룹 우선 + 그룹 안 모델 순서,0.3599
그룹 우선 + 그룹 안 저과거 순서,0.3981
그룹 우선 + 그룹 안 0.5모델+0.5저과거,0.3887


### 결과: 규칙 0.3 ≈ 0.5 > 모델 단독, 0.7·그룹 우선은 하락 → **LB 패턴(36≈38>21)과 일치** → 이 세트로 개선안 비교.

## 개선안 일괄 비교
- 두 번 전 라벨: 9/1에선 08-11 공식 라벨도 사용 가능 → 검증에선 07-28 라벨(재현, FUT18)로 같은 구조
- 세 번째 구성원: 분산 규칙(q75), 최근 모멘텀
- 모델 변형: C, 최근성 피처

In [6]:
old = make_labels('2026-07-28', fut=18)[['channel_id', 'target']].rename(columns={'target': 'prev2'})
va = va.drop(columns=[c for c in ['prev2'] if c in va.columns]).merge(old, on='channel_id', how='left')
q = con.sql("""SELECT channel_id, QUANTILE_CONT(LN(1 + view_5d), 0.75) q75 FROM video_5d_views
               WHERE is_shorts = 0 AND view_5d IS NOT NULL AND published_at < TIMESTAMP '2026-08-25' GROUP BY 1""").df()
va = va.drop(columns=[c for c in ['q75'] if c in va.columns]).merge(q, on='channel_id', how='left')
p2 = va.prev2.fillna(0.2).values
base = 0.5 * r(p_hist) + 0.5 * rule
res = {'B0 기준 (모델 0.5 + 규칙 0.5)': ap(base)}
print('두 번 전 라벨별 Rising:', va.groupby([va.prev.fillna(-1), va.prev2.fillna(-1)]).target.agg(['mean', 'size']).round(3).to_string())
# 두 번 전 라벨 반영
for a in [0.2, 0.4, 0.6]:
    rule2 = r(prev_f + a * p2 + 1e-3 * r(-va.log_past_med))
    res[f'B1 규칙에 두번전 라벨 a={a}'] = ap(0.5 * r(p_hist) + 0.5 * rule2)
res['B1b 두번전 라벨을 별도 구성원 0.15'] = ap(0.425 * r(p_hist) + 0.425 * rule + 0.15 * r(p2))
# 세 번째 구성원
qr = 0.8 * va.q75 - va.log_past_med
for w in [0.1, 0.2]:
    res[f'B2 + q75 규칙 {w}'] = ap((1 - w) * base + w * r(qr))
    res[f'B3 + mom_7 {w}'] = ap((1 - w) * base + w * r(va.mom_7))
    res[f'B3b + 저과거 {w}'] = ap((1 - w) * base + w * r(-va.log_past_med))
# 모델 변형
for C in [0.003, 0.03, 0.1]:
    pm = logreg(C).fit(tr[PRUNED], tr.target).predict_proba(va[PRUNED])[:, 1]
    res[f'B4 모델 C={C}'] = ap(0.5 * r(pm) + 0.5 * rule)
pm23 = logreg(0.1).fit(tr[FEATURES], tr.target).predict_proba(va[FEATURES])[:, 1]
res['B4 모델 23피처 C=0.1 (05 방식)'] = ap(0.5 * r(pm23) + 0.5 * rule)
pd.Series(res).sort_values(ascending=False).to_frame('PR-AUC (공식 08-25)')

두 번 전 라벨별 Rising:              mean  size
prev prev2             
-1.0 -1.0   0.273    66
 0.0 -1.0   0.203   138
      0.0   0.143   294
      1.0   0.128    47
 1.0 -1.0   0.420    50
      0.0   0.371    35
      1.0   0.167    36


,PR-AUC (공식 08-25)
B2 + q75 규칙 0.2,0.4457
B3b + 저과거 0.2,0.4371
B2 + q75 규칙 0.1,0.4330
B3b + 저과거 0.1,0.4294
B3 + mom_7 0.2,0.4230
B3 + mom_7 0.1,0.4219
B4 모델 23피처 C=0.1 (05 방식),0.4197
B4 모델 C=0.03,0.4175
B0 기준 (모델 0.5 + 규칙 0.5),0.4172
B4 모델 C=0.1,0.4156


### 결과
- **q75 규칙을 세 번째 구성원(0.2)으로: 0.4172 → 0.4457 (+0.029)**, 저과거 0.2: +0.020
- **두 번 전 라벨은 반대 방향**: 두 번 연속 라벨=1 채널의 Rising 0.167 (한 번만 1인 채널 0.37~0.42) → 평균회귀. 음(−)의 가중치로 시험.

## 확장 격자: q75 비중 × 두 번 전 라벨 음의 가중치

In [7]:
rows = []
for wq in [0.0, 0.15, 0.2, 0.3, 0.4]:
    for a in [0.0, -0.3, -0.6, -1.0]:
        rule_a = r(prev_f + a * (p2 * (va.prev == 1).fillna(False).values) + 1e-3 * r(-va.log_past_med))
        s = (1 - wq) * (0.5 * r(p_hist) + 0.5 * rule_a) + wq * r(qr)
        rows.append({'q75 비중': wq, '두번전 라벨(라벨=1일 때) 가중치': a, 'PR-AUC': ap(s)})
G = pd.DataFrame(rows).pivot(index='q75 비중', columns='두번전 라벨(라벨=1일 때) 가중치', values='PR-AUC')
print(G)
# 모델:규칙 비율도 q75와 함께
rows = []
for wm in [0.3, 0.5, 0.7]:
    for wq in [0.2, 0.3]:
        s = (1 - wq) * ((1 - wm) * rule + wm * r(p_hist)) + wq * r(qr)
        rows.append({'모델 비율': wm, 'q75': wq, 'PR-AUC': ap(s)})
print(pd.DataFrame(rows).pivot(index='모델 비율', columns='q75', values='PR-AUC'))

두번전 라벨(라벨=1일 때) 가중치    -1.0    -0.6    -0.3     0.0
q75 비중                                             
0.00                 0.4512  0.4273  0.4273  0.4172
0.15                 0.4771  0.4560  0.4560  0.4389
0.20                 0.4815  0.4618  0.4618  0.4457
0.30                 0.4840  0.4683  0.4683  0.4533
0.40                 0.4846  0.4715  0.4715  0.4568
q75       0.2     0.3
모델 비율                
0.3    0.4398  0.4489
0.5    0.4457  0.4533
0.7    0.4424  0.4485


### 결과: **q75 0.3 + 두 번 연속 라벨=1 채널을 라벨=0 취급 → 0.4172 → 0.4840 (+0.067)**
두 번 연속 Rising 판정 채널 = 이미 두 번 떠서 과거 기준선이 높아짐 → 평균회귀.

## 두 번째 검증 지점: 정답 = 공식 08-11, 전주 = 07-28(재현), 두 번 전 = 07-14(재현)
모델은 정답 구간이 08-11 이전에 끝나는 자체 라벨(07-18~07-24, FUT18)로 학습.

In [8]:
va11 = tl[tl.ref_date == '2026-08-11'][['channel_id', 'target']].merge(make_features('2026-08-11'), on='channel_id', how='left')
va11 = va11.merge(make_labels('2026-07-28', fut=18)[['channel_id', 'target']].rename(columns={'target': 'prev'}), on='channel_id', how='left')
va11 = va11.merge(make_labels('2026-07-14', fut=18)[['channel_id', 'target']].rename(columns={'target': 'prev2'}), on='channel_id', how='left')
q11 = con.sql("""SELECT channel_id, QUANTILE_CONT(LN(1 + view_5d), 0.75) q75 FROM video_5d_views
                 WHERE is_shorts = 0 AND view_5d IS NOT NULL AND published_at < TIMESTAMP '2026-08-11' GROUP BY 1""").df()
va11 = va11.merge(q11, on='channel_id', how='left')
tr11 = attach_features(pd.concat([make_labels(d, fut=18) for d in pd.date_range('2026-07-18', '2026-07-24').strftime('%Y-%m-%d')], ignore_index=True))
N11 = len(va11); r11 = lambda x: rankdata(pd.Series(np.asarray(x, float)).fillna(np.nanmedian(x))) / N11
y11 = va11.target.values; ap11 = lambda s: round(average_precision_score(y11, s), 4)
ph11 = logreg(0.01).fit(tr11[PRUNED], tr11.target).predict_proba(va11[PRUNED])[:, 1]
pf11 = va11.prev.fillna(0.2).values; pp11 = va11.prev2.fillna(0.2).values
print('행', N11, '| 전주 라벨 1/0/없음', int((va11.prev == 1).sum()), int((va11.prev == 0).sum()), int(va11.prev.isna().sum()),
      '| 두번전 있음', int(va11.prev2.notna().sum()))
print(va11.groupby([va11.prev.fillna(-1), va11.prev2.fillna(-1)]).target.agg(['mean', 'size']).round(3).to_string())
qr11 = 0.8 * va11.q75 - va11.log_past_med
rows = []
for wq in [0.0, 0.2, 0.3]:
    for a in [0.0, -1.0]:
        ru = r11(pf11 + a * (pp11 * (va11.prev == 1).fillna(False).values) + 1e-3 * r11(-va11.log_past_med))
        rows.append({'q75': wq, '두번전 가중치': a, 'PR-AUC': ap11((1 - wq) * (0.5 * r11(ph11) + 0.5 * ru) + wq * r11(qr11))})
print(pd.DataFrame(rows).pivot(index='q75', columns='두번전 가중치', values='PR-AUC'))

행 601 | 전주 라벨 1/0/없음 83 329 189 | 두번전 있음 58
             mean  size
prev prev2             
-1.0 -1.0   0.265   189
 0.0 -1.0   0.114   281
      0.0   0.050    40
      1.0   0.125     8
 1.0 -1.0   0.466    73
      0.0   0.167     6
      1.0   0.250     4
두번전 가중치    -1.0     0.0
q75                    
0.0      0.4335  0.4206
0.2      0.4480  0.4403
0.3      0.4464  0.4405


### 결과 (두 번째 지점, 정답 08-11): 두 번 전 라벨 −1.0 → +0.008~0.013 (해당 채널 4개뿐), q75 0.2 → +0.02. **두 검증 지점에서 방향 일치.**

## submit_45: submit_38 + 두 번 연속 라벨=1 강등 + q75 0.25
- rule_a = rank(8/25 라벨 − 1.0·[8/11 라벨=1 & 8/25 라벨=1] + 1e-3·rank(−과거 원값))
- base = 0.5·rank(submit_21) + 0.5·rule_a
- 최종 = 0.75·base + 0.25·rank(0.8·q75(급등 보정값) − log 과거 중앙값(원값))

In [9]:
sample = pd.read_csv(f'{RAW}/sample_submission.csv', encoding='utf-8-sig')
sample['channel_id'] = sample.row_id.str.rsplit('_', n=1).str[0]
T = sample.merge(tl[tl.ref_date == '2026-08-25'][['channel_id', 'target']].rename(columns={'target': 'prev'}), on='channel_id', how='left')
T = T.merge(tl[tl.ref_date == '2026-08-11'][['channel_id', 'target']].rename(columns={'target': 'prev2'}), on='channel_id', how='left')
con.sql("CREATE OR REPLACE TABLE dl AS SELECT * FROM (VALUES " + ",".join(
    f"(DATE '2026-08-{d:02d}', {s})" for d, s in [(24, .045), (25, .092), (26, .326), (27, .661), (28, .759), (29, .804), (30, .789), (31, .782)]) + ") t(day, s)")
pm = con.sql("""SELECT channel_id, MEDIAN(LN(1 + view_5d)) lp_raw,
                       QUANTILE_CONT(LN(1 + view_5d) - COALESCE(dl.s, 0), 0.75) q75_def, QUANTILE_CONT(LN(1 + view_5d), 0.75) q75_raw
                FROM video_5d_views LEFT JOIN dl ON CAST(published_at AS DATE) = dl.day
                WHERE is_shorts = 0 AND view_5d IS NOT NULL AND published_at < TIMESTAMP '2026-09-01' GROUP BY 1""").df()
T = T.merge(pm, on='channel_id', how='left')
T['s21'] = pd.read_csv('../submissions/submit_21_ens_s4_s3_s15.csv').set_index('row_id').prediction.loc[T.row_id].values
T['s38'] = pd.read_csv('../submissions/submit_38_ens21_prevrule50.csv').set_index('row_id').prediction.loc[T.row_id].values
NT = len(T); rt = lambda x: rankdata(pd.Series(np.asarray(x, float)).fillna(np.nanmedian(x))) / NT
dbl = ((T.prev == 1) & (T.prev2 == 1)).values
print('8/25 라벨=1:', int((T.prev == 1).sum()), '| 그중 8/11도 1 (강등 대상):', int(dbl.sum()))
pf = T.prev.fillna(0.2).values - 1.0 * dbl
rule_a = rt(pf + 1e-3 * rt(-T.lp_raw))
base = 0.5 * rt(T.s21) + 0.5 * rule_a
qr9 = 0.8 * T.q75_def - T.lp_raw
s45 = 0.75 * base + 0.25 * rt(qr9) + 1e-7 * rt(T.s21)
sv = rankdata(s45) / NT; sv = (sv - sv.min()) / (sv.max() - sv.min())
sub = pd.DataFrame({'row_id': T.row_id, 'prediction': sv})
assert list(sub.columns) == ['row_id', 'prediction'] and len(sub) == 694 and sub.row_id.is_unique
assert set(sub.row_id) == set(sample.row_id) and sub.prediction.between(0, 1).all() and not sub.prediction.duplicated().any()
sub.to_csv('../submissions/submit_45_ens38_dblDemote_q75.csv', index=False, encoding='utf-8')
top38, top45 = set(np.argsort(-T.s38.values)[:139]), set(np.argsort(-s45)[:139])
print('38과 순위상관', round(spearmanr(T.s38, s45)[0], 4), '| 상위139 중 바뀐 수', len(top38 - top45),
      '| 45 상위139 중 8/25 라벨=1', int((T.prev.values[list(top45)] == 1).sum()), '| 그중 강등 대상', int(dbl[list(top45)].sum()),
      '| 38 상위139 중 강등 대상', int(dbl[list(top38)].sum()))

8/25 라벨=1: 134 | 그중 8/11도 1 (강등 대상): 40
38과 순위상관 0.9757 | 상위139 중 바뀐 수 31 | 45 상위139 중 8/25 라벨=1 99 | 그중 강등 대상 19 | 38 상위139 중 강등 대상 36


## submit_45 디벨롭 + 랜덤포레스트 검토 (두 검증 지점 모두에서 오르는 변형만 채택)
- 지점 A: 정답 08-25(공식), 전주 08-11(공식), 두 번 전 07-28(재현)
- 지점 B: 정답 08-11(공식), 전주 07-28(재현), 두 번 전 07-14(재현)

In [10]:
from sklearn.ensemble import RandomForestClassifier
def rf_fit_pred(trd, vad, seed=0):
    imp = SimpleImputer(strategy='median').fit(trd[PRUNED])
    m = RandomForestClassifier(n_estimators=400, min_samples_leaf=40, max_features=0.5, n_jobs=-1, random_state=seed)
    return m.fit(imp.transform(trd[PRUNED]), trd.target).predict_proba(imp.transform(vad[PRUNED]))[:, 1]
pts = {
  'A': dict(v=va, y=y, rr=r, ph=p_hist, prf=rf_fit_pred(tr, va), pf=prev_f, p2=p2, qr=qr),
  'B': dict(v=va11, y=y11, rr=r11, ph=ph11, prf=rf_fit_pred(tr11, va11), pf=pf11, p2=pp11, qr=qr11)}
def score(P, wm=0.5, a=-1.0, wq=0.25, wlow=0.0, model='lr', wrf=0.0):
    v, rr = P['v'], P['rr']
    dbl = ((v.prev == 1) & (v.prev2 == 1)).values
    ru = rr(P['pf'] + a * dbl + 1e-3 * rr(-v.log_past_med))
    m = rr(P['ph']) if model == 'lr' else rr(P['prf'])
    if wrf: m = (1 - wrf) * rr(P['ph']) + wrf * rr(P['prf'])
    base = wm * m + (1 - wm) * ru
    return (1 - wq - wlow) * base + wq * rr(P['qr']) + wlow * rr(-v.log_past_med)
def ev(**kw):
    return {k: round(average_precision_score(P['y'], score(P, **kw)), 4) for k, P in pts.items()}
rows = {'45 구성 (기준)': ev()}
rows['RF 단독 모델 (45 구조)'] = ev(model='rf')
rows['LR+RF 반반 모델 (45 구조)'] = ev(wrf=0.5)
rows['LR+RF 0.3'] = ev(wrf=0.3)
for wq in [0.15, 0.2, 0.3, 0.35]:
    rows[f'q75 {wq}'] = ev(wq=wq)
for wm in [0.4, 0.6]:
    rows[f'모델 비율 {wm}'] = ev(wm=wm)
for a in [-0.6, -2.0]:
    rows[f'강등 강도 {a}'] = ev(a=a)
rows['+ 저과거 0.1'] = ev(wlow=0.1)
rows['q75 0.2 + 저과거 0.1'] = ev(wq=0.2, wlow=0.1)
R = pd.DataFrame(rows).T
R['A 변화'] = (R.A - R.loc['45 구성 (기준)', 'A']).round(4); R['B 변화'] = (R.B - R.loc['45 구성 (기준)', 'B']).round(4)
R['둘 다 오름'] = (R['A 변화'] > 0) & (R['B 변화'] > 0)
print('참고 — 모델 단독 PR-AUC  LR A', round(average_precision_score(y, p_hist), 4), 'RF A', round(average_precision_score(y, pts['A']['prf']), 4),
      '| LR B', round(average_precision_score(y11, ph11), 4), 'RF B', round(average_precision_score(y11, pts['B']['prf']), 4))
print(R.to_string())

참고 — 모델 단독 PR-AUC  LR A 0.4005 RF A 0.3337 | LR B 0.327 RF B 0.3113
                          A       B    A 변화    B 변화  둘 다 오름
45 구성 (기준)           0.4834  0.4487  0.0000  0.0000   False
RF 단독 모델 (45 구조)     0.4411  0.4526 -0.0423  0.0039   False
LR+RF 반반 모델 (45 구조)  0.4688  0.4549 -0.0146  0.0062   False
LR+RF 0.3            0.4765  0.4526 -0.0069  0.0039   False
q75 0.15             0.4767  0.4474 -0.0067 -0.0013   False
q75 0.2              0.4804  0.4491 -0.0030  0.0004   False
q75 0.3              0.4821  0.4480 -0.0013 -0.0007   False
q75 0.35             0.4822  0.4454 -0.0012 -0.0033   False
모델 비율 0.4            0.4850  0.4582  0.0016  0.0095    True
모델 비율 0.6            0.4772  0.4403 -0.0062 -0.0084   False
강등 강도 -0.6           0.4637  0.4441 -0.0197 -0.0046   False
강등 강도 -2.0           0.4864  0.4496  0.0030  0.0009    True
+ 저과거 0.1            0.4748  0.4443 -0.0086 -0.0044   False
q75 0.2 + 저과거 0.1    0.4765  0.4459 -0.0069 -0.0028   False


### 결과: RF는 지점 A에서 크게 하락(−0.042) → 제외. 두 지점 모두 오른 변형: 모델 비율 0.4(+0.002/+0.010), 강등 강도 −2.0(+0.003/+0.001).
### 결합 확인 → 둘 다 오르면 submit_46

In [11]:
combo = ev(wm=0.4, a=-2.0)
print('결합 (모델 0.4, 강등 −2.0):', combo, '| 45 기준:', rows['45 구성 (기준)'])
if combo['A'] > rows['45 구성 (기준)']['A'] and combo['B'] > rows['45 구성 (기준)']['B']:
    pf46 = T.prev.fillna(0.2).values - 2.0 * dbl
    rule46 = rt(pf46 + 1e-3 * rt(-T.lp_raw))
    s46 = 0.75 * (0.4 * rt(T.s21) + 0.6 * rule46) + 0.25 * rt(qr9) + 1e-7 * rt(T.s21)
    sv = rankdata(s46) / NT; sv = (sv - sv.min()) / (sv.max() - sv.min())
    sub = pd.DataFrame({'row_id': T.row_id, 'prediction': sv})
    assert len(sub) == 694 and sub.row_id.is_unique and set(sub.row_id) == set(sample.row_id)
    assert sub.prediction.between(0, 1).all() and not sub.prediction.duplicated().any()
    sub.to_csv('../submissions/submit_46_ens38_dbl2_q75_rule60.csv', index=False, encoding='utf-8')
    top46 = set(np.argsort(-s46)[:139])
    print('submit_46 저장 | 45와 순위상관', round(spearmanr(s45, s46)[0], 4), '| 상위139 중 45와 다른 수', len(top45 - top46),
          '| 상위139 중 강등 대상', int(dbl[list(top46)].sum()))
else:
    print('결합 시 한 지점이라도 하락 → submit_45 유지')

결합 (모델 0.4, 강등 −2.0): {'A': 0.4863, 'B': 0.459} | 45 기준: {'A': 0.4834, 'B': 0.4487}
submit_46 저장 | 45와 순위상관 0.9615 | 상위139 중 45와 다른 수 19 | 상위139 중 강등 대상 0


### submit_46 저장 (모델 0.4 + 강등 −2.0): 지점 A 0.4863, B 0.4590 — 두 지점 모두 45보다 높음.

## 추가 디벨롭: "이미 얼마나 올랐나"를 연속값으로
두 번 전 기준일의 실제 growth(재현, FUT18)를 사용. 라벨=1 그룹 안에서 이미 많이 오른 채널일수록 감점.
지점 A: 07-28 growth / 지점 B: 07-14 growth / 테스트: 08-11 growth

In [12]:
def growth_at(Dref, fut=18):
    return con.sql(f"""
      WITH a AS (SELECT channel_id,
                   MEDIAN(view_5d) FILTER (WHERE published_at < TIMESTAMP '{Dref}') ps,
                   COUNT(view_5d) FILTER (WHERE published_at < TIMESTAMP '{Dref}') n,
                   COALESCE(MEDIAN(view_5d) FILTER (WHERE published_at >= TIMESTAMP '{Dref}' AND published_at < TIMESTAMP '{Dref}' + INTERVAL {fut} DAY), 0) fs
                 FROM video_5d_views WHERE is_shorts = 0 AND view_5d IS NOT NULL GROUP BY 1)
      SELECT channel_id, LN(1 + fs) - LN(1 + ps) AS g2 FROM a WHERE n >= 3 AND ps >= 100""").df()
for key, d2 in [('A', '2026-07-28'), ('B', '2026-07-14')]:
    v = pts[key]['v']
    pts[key]['v'] = v.drop(columns=[c for c in ['g2'] if c in v.columns]).merge(growth_at(d2), on='channel_id', how='left')
def score2(P, wm=0.4, a=-2.0, wq=0.25, b=0.0, use_bin=True):
    v, rr = P['v'], P['rr']
    dbl = ((v.prev == 1) & (v.prev2 == 1)).values if use_bin else np.zeros(len(v), bool)
    g2r = rr(v.g2.fillna(v.g2.median()))                                # 이미 오른 정도 (0~1)
    pen = b * g2r * (v.prev == 1).fillna(False).values
    ru = rr(P['pf'] + a * dbl - pen + 1e-3 * rr(-v.log_past_med))
    base = wm * rr(P['ph']) + (1 - wm) * ru
    return (1 - wq) * base + wq * rr(P['qr'])
def ev2(**kw):
    return {k: round(average_precision_score(P['y'], score2(P, **kw)), 4) for k, P in pts.items()}
base46 = ev2()
rows2 = {'46 구성 (기준)': base46}
for b in [0.3, 0.6, 1.0]:
    rows2[f'46 + 연속 감점 b={b}'] = ev2(b=b)
    rows2[f'연속 감점만 b={b} (이진 강등 없음)'] = ev2(b=b, use_bin=False)
R2 = pd.DataFrame(rows2).T
R2['A 변화'] = (R2.A - base46['A']).round(4); R2['B 변화'] = (R2.B - base46['B']).round(4)
print(R2.to_string())

                              A       B    A 변화    B 변화
46 구성 (기준)               0.4863  0.4590  0.0000  0.0000
46 + 연속 감점 b=0.3         0.4905  0.4549  0.0042 -0.0041
연속 감점만 b=0.3 (이진 강등 없음)  0.4702  0.4513 -0.0161 -0.0077
46 + 연속 감점 b=0.6         0.4905  0.4549  0.0042 -0.0041
연속 감점만 b=0.6 (이진 강등 없음)  0.4699  0.4517 -0.0164 -0.0073
46 + 연속 감점 b=1.0         0.4926  0.4558  0.0063 -0.0032
연속 감점만 b=1.0 (이진 강등 없음)  0.4825  0.4550 -0.0038 -0.0040


### 연속 감점: 지점 A만 오르고 B는 하락 → 제외. submit_46 유지.

## 세 번째 독립 근거: 급등 전 학습 쌍에서도 "두 번 연속 라벨=1 → Rising 낮음"인가?
기준일 D(07-26~08-05, FUT26 정답)마다 D−7 라벨(FUT18)과 D−14 라벨(FUT18)을 재현해 그룹별 Rising 비율 비교.

In [13]:
from datetime import datetime, timedelta
sh = lambda d, k: (datetime.strptime(d, '%Y-%m-%d') + timedelta(days=k)).strftime('%Y-%m-%d')
rows = []
for D_ in pd.date_range('2026-07-26', '2026-08-05').strftime('%Y-%m-%d'):
    cur = make_labels(D_, fut=26)[['channel_id', 'target']]
    p1 = make_labels(sh(D_, -7), fut=18)[['channel_id', 'target']].rename(columns={'target': 'prev'})
    p2_ = make_labels(sh(D_, -14), fut=18)[['channel_id', 'target']].rename(columns={'target': 'prev2'})
    rows.append(cur.merge(p1, on='channel_id', how='left').merge(p2_, on='channel_id', how='left').assign(D=D_))
pp = pd.concat(rows, ignore_index=True)
g = pp[pp.prev2.notna() & pp.prev.notna()].groupby(['prev', 'prev2']).target.agg(['mean', 'size']).round(3)
print('급등 전 학습 쌍 (D−7·D−14 라벨 모두 있는 행):'); print(g.to_string())
print('라벨=1 중: 두 번 연속 1 →', g.loc[(1.0, 1.0), 'mean'], '| 직전만 1 →', g.loc[(1.0, 0.0), 'mean'])

급등 전 학습 쌍 (D−7·D−14 라벨 모두 있는 행):
             mean  size
prev prev2             
0.0  0.0    0.073   803
     1.0    0.147   102
1.0  0.0    0.643   115
     1.0    0.485   136
라벨=1 중: 두 번 연속 1 → 0.485 | 직전만 1 → 0.643


### 결과: 세 설정 모두 "두 번 연속 라벨=1 → 덜 뜬다" 방향 일치. 단 급등 전 학습 쌍(본선과 같은 7일 간격)에선 두 번 연속 채널 Rising 0.485로 라벨=0(0.07~0.15)보다 훨씬 높음
→ submit_46처럼 맨 아래로 내리면 본선이 학습 쌍처럼 움직일 경우 정답 ~19개를 버릴 위험.

## submit_47: 46의 개선(모델 0.4, q75 0.25) 유지 + 강등은 라벨=0 수준(−1.0)

In [14]:
print('47 구성 검증:', ev(wm=0.4, a=-1.0), '| 45:', ev(), '| 46:', ev(wm=0.4, a=-2.0))
pf47 = T.prev.fillna(0.2).values - 1.0 * dbl
rule47 = rt(pf47 + 1e-3 * rt(-T.lp_raw))
s47 = 0.75 * (0.4 * rt(T.s21) + 0.6 * rule47) + 0.25 * rt(qr9) + 1e-7 * rt(T.s21)
sv = rankdata(s47) / NT; sv = (sv - sv.min()) / (sv.max() - sv.min())
sub = pd.DataFrame({'row_id': T.row_id, 'prediction': sv})
assert len(sub) == 694 and sub.row_id.is_unique and set(sub.row_id) == set(sample.row_id)
assert sub.prediction.between(0, 1).all() and not sub.prediction.duplicated().any()
sub.to_csv('../submissions/submit_47_ens38_dbl1_q75_rule60.csv', index=False, encoding='utf-8')
top47 = set(np.argsort(-s47)[:139])
print('submit_47 | 상위139 중 강등 대상', int(dbl[list(top47)].sum()), '(45: 19, 46: 0, 38: 36)',
      '| 38과 상위139 겹침', len(top38 & top47), '| 38과 순위상관', round(spearmanr(T.s38, s47)[0], 4))

47 구성 검증: {'A': 0.485, 'B': 0.4582} | 45: {'A': 0.4834, 'B': 0.4487} | 46: {'A': 0.4863, 'B': 0.459}
submit_47 | 상위139 중 강등 대상 18 (45: 19, 46: 0, 38: 36) | 38과 상위139 겹침 107 | 38과 순위상관 0.9677
